In [ ]:
import json
from pathlib import Path
from collections import Counter

In [1]:
DATASET = Path("output/normalized_dataset.json")

with open(DATASET, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Dataset loaded successfully.")

NameError: name 'Path' is not defined

In [ ]:
print("========== DATASET OVERVIEW ==========\n")

print("Logs:", len(data["logs"]))
print("Uploads:", len(data["uploads"]))
print("Applications:", len(data["summary"]["identified_applications"]))
print("Users:", len(data["summary"]["identified_usernames"]))

In [ ]:
print("========== MISSING VALUES ==========\n")

missing = {}

for section in ["client","collection","requests"]:

    missing[section] = {}

    for key,value in data[section].items():

        if value is None:

            missing[section][key] = "Missing"

print(missing)

In [ ]:
print("========== UPLOAD MISSING VALUES ==========\n")

missing_counter = Counter()

for upload in data["uploads"]:

    for key,value in upload.items():

        if value in [None,"",[]]:

            missing_counter[key]+=1

for key,value in sorted(missing_counter.items()):

    print(f"{key}: {value}")

In [ ]:
print("========== LOG MISSING VALUES ==========\n")

missing_counter = Counter()

for log in data["logs"]:

    for key,value in log.items():

        if value in [None,"",[]]:

            missing_counter[key]+=1

for key,value in sorted(missing_counter.items()):

    print(f"{key}: {value}")

In [ ]:
print("========== DUPLICATE UPLOADS ==========\n")

seen=set()

duplicates=[]

for upload in data["uploads"]:

    key=(

        upload["path"],

        upload["timestamp"],

        upload["file_size"]

    )

    if key in seen:

        duplicates.append(upload)

    else:

        seen.add(key)

print("Duplicates:",len(duplicates))

In [ ]:
print("========== DUPLICATE LOGS ==========\n")

seen=set()

duplicates=[]

for log in data["logs"]:

    key=(

        log["timestamp"],

        log["message"]

    )

    if key in seen:

        duplicates.append(log)

    else:

        seen.add(key)

print("Duplicates:",len(duplicates))

In [ ]:
print("========== UPLOAD STATUS ==========\n")

complete=0

incomplete=0

unknown=0

for upload in data["uploads"]:

    status=upload["upload_complete"]

    if status==True:

        complete+=1

    elif status==False:

        incomplete+=1

    else:

        unknown+=1

print("Complete:",complete)
print("Incomplete:",incomplete)
print("Unknown:",unknown)

In [ ]:
print("========== ZERO BYTE FILES ==========\n")

zero=[]

for upload in data["uploads"]:

    if upload["file_size"]==0:

        zero.append(upload)

print("Zero byte files:",len(zero))

In [ ]:
print("========== TIMESTAMP QUALITY ==========\n")

missing=0

for upload in data["uploads"]:

    if upload["timestamp"] is None:

        missing+=1

print("Uploads without timestamp:",missing)

missing=0

for log in data["logs"]:

    if log["timestamp"] is None:

        missing+=1

print("Logs without timestamp:",missing)

In [ ]:
print("========== USERNAMES ==========\n")

counter=Counter()

for upload in data["uploads"]:

    username=upload["username"]

    if username:

        counter[username]+=1

print(counter)

In [ ]:
print("========== APPLICATIONS ==========\n")

counter=Counter()

for upload in data["uploads"]:

    app=upload["application"]

    if app:

        counter[app]+=1

print(counter)

In [ ]:
print("========== FILE EXTENSIONS ==========\n")

counter=Counter()

for upload in data["uploads"]:

    ext=upload["extension"]

    counter[ext]+=1

print(counter.most_common(30))

In [ ]:
print("========== SUSPICIOUS UPLOADS ==========\n")

suspicious=[]

for upload in data["uploads"]:

    if upload["upload_complete"]==False:

        suspicious.append(upload)

print("Suspicious uploads:",len(suspicious))

In [ ]:
print("========== FLAGGING DUPLICATE UPLOADS ==========\n")

seen = set()
duplicate_count = 0

for upload in data["uploads"]:

    key = (
        upload["path"],
        upload["timestamp"],
        upload["file_size"]
    )

    if key in seen:
        upload["is_duplicate"] = True
        duplicate_count += 1
    else:
        upload["is_duplicate"] = False
        seen.add(key)

print("Duplicate uploads found:", duplicate_count)

In [ ]:
processed_dataset={

    "dataset_metadata":data["dataset_metadata"],

    "summary":data["summary"],

    "client":data["client"],

    "collection":data["collection"],

    "requests":data["requests"],

    "logs":data["logs"],

    "uploads":clean_uploads

}

In [ ]:
OUTPUT=Path("output/processed_dataset.json")

with open(OUTPUT,"w",encoding="utf-8") as f:

    json.dump(

        processed_dataset,

        f,

        indent=2,

        ensure_ascii=False

    )

print("Processed dataset saved.")